# Cross-checking classical logic with native tensor networks

Adapted from the Qualtran v0.7.0 `xcheck_classical_quimb` notebook (Copyright 2023 Google LLC, Apache-2.0).
[Locked source](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/simulation/xcheck_classical_quimb.ipynb).

This notebook follows the same 13-cell workflow using FlagQuantum graphs, tensor suppliers and drawings. Run from the repository environment with the repository root on `PYTHONPATH`.

Classical callbacks predict output basis values. Independent tensor contraction then checks the complex overlap. An overlap of one confirms the chosen classical input/output pair; zero rejects it. The explicit cotengra plan is example policy, with a budget of 1,024 peak tensor elements. It does not report process RAM.

In [ ]:
import torch
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from examples.algorithms.classical_tensor_crosscheck import prepare_classical_crosscheck
from examples.algorithms.classical_tensor_networks import (
    closed_crosscheck_network,
    contract_closed_crosscheck,
    select_crosscheck_graph,
)
from examples.algorithms.classical_tensor_scores import crosscheck_score_data
from flagquantum.core.ir.program import ProgramRecord
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.simulation.tensor_network.contraction import (
    _contract_nodes_with_slicing_plan,
)


def show_graph(
    record: ProgramRecord, kind: str, **options: float
) -> tuple[Figure, Axes]:
    return draw_register_score(crosscheck_score_data(record, kind), **options)


## X

In [ ]:
kind = "xgate"


Starting in zero must produce one. The graph contains input preparation, the executable X body, and the predicted output effect.

In [ ]:
x_tt = prepare_classical_crosscheck(kind, {"q": 0})
assert contract_closed_crosscheck(x_tt, kind).item() == 1.0
show_graph(x_tt, kind)


Explicit output values bypass classical inference. An incorrect zero output for this input contracts to zero.

In [ ]:
wrong_x = prepare_classical_crosscheck("xgate", {"q": 0}, {"q": 0})
result = contract_closed_crosscheck(wrong_x, "xgate")
assert result.item() == 0
if result.item() == 0:
    print("Tensor contracted to 0. The universe has been annihilated.")


## CNOT

Both control values and the target use the existing row-major, most-significant-bit-first register convention.

In [ ]:
kind = "cnot"


In [ ]:
cnot_tt = prepare_classical_crosscheck(kind, {"ctrl": 1, "target": 0})
assert contract_closed_crosscheck(cnot_tt, kind).item() == 1.0
show_graph(cnot_tt, kind)


## Toffoli

In [ ]:
kind = "toffoli"


In [ ]:
tof_tt = prepare_classical_crosscheck(kind, {"ctrl": [1, 1], "target": 0})
assert contract_closed_crosscheck(tof_tt, kind).item() == 1.0
show_graph(tof_tt, kind)


## Addition

In [ ]:
kind, width = "add", 5


In [ ]:
add_tt = prepare_classical_crosscheck(kind, {"a": 2, "b": 3}, width=width)
show_graph(add_tt, kind)


Select the existing factorized CNOT, AND, adjoint and word-rebinding tensor leaves. Logical tensor selection preserves work ownership; it is not physical measurement flattening.

In [ ]:
add_flat = select_crosscheck_graph(add_tt, "add")
show_graph(add_flat, "add", max_width=20, max_height=100)


In [ ]:
torch.testing.assert_close(
    contract_closed_crosscheck(add_tt, "add"), torch.tensor(1, dtype=torch.complex128)
)
torch.testing.assert_close(
    contract_closed_crosscheck(add_flat, "add"), torch.tensor(1, dtype=torch.complex128)
)


### Large bit width

Add(32) acts on 64 logical gate bits without a whole-word matrix. Each numerical leaf has at most 32 elements. Planned peak elements exclude other live tensors, interpreter and planner memory. Physical primitive counts and upstream resource models are documented separately in `docs/development/CLASSICAL_TENSOR_NOTEBOOK.md`.

In [ ]:
add_large_tt = prepare_classical_crosscheck("add", {"a": 5555, "b": 6666}, width=32)
add_large_flat = select_crosscheck_graph(add_large_tt, "add")
nodes, plan = closed_crosscheck_network(add_large_flat, "add")
print(
    f"64 logical gate bits; {len(nodes)} nodes; planned peak {plan.peak_size} complex elements; {plan.n_slices} slices."
)
large_result = _contract_nodes_with_slicing_plan(nodes, (), plan)
torch.testing.assert_close(large_result, torch.tensor(1, dtype=torch.complex128))
large_result
